In [1]:
print("Notebook is working")

Notebook is working


In [2]:
%pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 41.6 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 48.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

print(pd.__version__)

3.1.0


In [4]:
%pip install datasets pandas

  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 33.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 19.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 47.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 545.8/545.8 kB 28.1 MB/s  0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 36.0 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26/26 [datasets]/26 [datasets]ce-hub]]

[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "RevolutionCrossroads/nara_revolutionary_war_pension_files_PDFs",
    split="train",
    streaming=True
)

sample = list(dataset.take(5))
df = pd.DataFrame(sample)

df[["NAID", "title", "numberOfPages"]]

/Users/catietaras/Desktop/smithsonian_pensions/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,NAID,title,numberOfPages
0,111403815,Revolutionary War Pension and Bounty Land Warr...,5
1,111403821,Revolutionary War Pension and Bounty Land Warr...,18
2,111403840,Revolutionary War Pension and Bounty Land Warr...,33
3,111403874,Revolutionary War Pension and Bounty Land Warr...,10
4,111403885,Revolutionary War Pension and Bounty Land Warr...,43


In [11]:
record = sample[0]

print("TITLE:", record["title"])
print("SOURCE:", record["naraURL"])

pages = record.get("RevX_OCR_text") or []

for page_number, text in enumerate(pages[:3], start=1):
    print(f"\n--- Page {page_number} ---\n")
    print(text)

TITLE: Revolutionary War Pension and Bounty Land Warrant Application File B. L. Wt. 2,153-400, Joseph Torrey, Continental
SOURCE: https://catalog.archives.gov/id/111403815

--- Page 1 ---

SERVICE
Continental ✓
★ Torrey, Joseph
NUMBER
B.L.Wt. 2153-400
CONTENTS
Empty lined area for left column of contents
Empty lined area for right column of contents

--- Page 2 ---

81 Avenue 6 th

--- Page 3 ---

State of New York City & County of New York Surrogate's Office June the 2nd 1836
I certify that satisfactory evidence was adduced to me to prove that John Torrey, William Torrey, Joseph Torrey, James W Torrey and Edward P. Torrey
are the grand children and only heirs at law in fee of Joseph Torrey who was a Major in the Col. Hazen's Regiment of the Revolutionary Army -
Given under my hand and Seal of Office, this third June day of June 1836.
James Campbell
The above form will answer for the following States, viz.
Maine, Massachusetts, New-Hampshire, Rhode Island, Connecticut, Vermont } Judge 

In [10]:
rows = []

for record in sample:
    texts = record.get("RevX_OCR_text") or []
    numbers = record.get("pageNumber") or []
    links = record.get("pageLink") or []

    for i, text in enumerate(texts):
        rows.append({
            "file_id": record["NAID"],
            "file_title": record["title"],
            "page_number": numbers[i] if i < len(numbers) else None,
            "page_url": links[i] if i < len(links) else None,
            "text_original": text
        })

pages_df = pd.DataFrame(rows)

print("Page table created:", len(pages_df), "rows")

Page table created: 109 rows


In [12]:
widow_pages = pages_df[
    pages_df["text_original"].str.contains(
        r"\bwidow\b", case=False, na=False
    )
]

print("Pages mentioning widow:", len(widow_pages))

for _, page in widow_pages.head(3).iterrows():
    print("\nFILE:", page["file_title"])
    print("PAGE:", page["page_number"])
    print("SOURCE:", page["page_url"])
    print(page["text_original"])

Pages mentioning widow: 23

FILE: Revolutionary War Pension and Bounty Land Warrant Application File S. 42501, Asa Torrey, Conn.
PAGE: 14
SOURCE: https://catalog.archives.gov/id/111403821?objectPage=14
Information as contained in his (or his widow's) application for pension on file in this Bureau. Asa Terry Inv. File 42,501
   DATES OF ENLISTMENT OR APPOINTMENT. LENGTH OF SERVICE. RANK. OFFICERS UNDER WHOM SERVICE WAS RENDERED. STATE.   CAPTAIN. COLONEL.     Jan 1 or Feb 1 1777 Dec 31 1776 Isaac Hyde Samuel Wyllys Conn.   1777 commanded a company of Artificers   1778 served to end of the war        
Battles engaged in, _____
Residence of soldier at enlistment, enlisted in Norwich Conn.
Date of application for pension, Apr. 29 1818 His claim as al.
Residence at date of application, Boston, Niagara Co. N.Y.
Age at date of application, 58 yrs.
Remarks: In 1820 children were Ambrose 20 years old; George 14 years; Delia Ann 12 years; Alvira 10 years. The name Asa Terry does not appear on th

In [13]:
terms = ["widow of", "per annum", "rate of", "dollars"]

def matches_widow_pension(record):
    pages = record.get("RevX_OCR_text") or []
    text = " ".join(
        page for page in pages if isinstance(page, str)
    )
    text = " ".join(text.lower().split())

    return all(term in text for term in terms)

test_matches = [
    record for record in sample
    if matches_widow_pension(record)
]

print("Files searched:", len(sample))
print("Candidate files:", len(test_matches))

Files searched: 5
Candidate files: 1


In [14]:
from pathlib import Path
import json

output_folder = Path("search_results")
output_folder.mkdir(exist_ok=True)

output_file = output_folder / "widow_pension_candidates.jsonl"

searched = 0
matched = 0

with output_file.open("w", encoding="utf-8") as f:
    for record in dataset:
        searched += 1

        if matches_widow_pension(record):
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
            matched += 1

        if searched % 500 == 0:
            f.flush()
            print(
                f"Searched {searched:,} files | "
                f"Candidates: {matched:,}",
                flush=True
            )

print(f"Finished: {searched:,} files searched; {matched:,} candidates.")
print("Saved to:", output_file.resolve())

Searched 500 files | Candidates: 167
Searched 1,000 files | Candidates: 312
Searched 1,500 files | Candidates: 457
Searched 2,000 files | Candidates: 607
Searched 2,500 files | Candidates: 752
Searched 3,000 files | Candidates: 898
Searched 3,500 files | Candidates: 1,065
Searched 4,000 files | Candidates: 1,251
Searched 4,500 files | Candidates: 1,408
Searched 5,000 files | Candidates: 1,554
Searched 5,500 files | Candidates: 1,719
Searched 6,000 files | Candidates: 1,878
Searched 6,500 files | Candidates: 2,037
Searched 7,000 files | Candidates: 2,198
Searched 7,500 files | Candidates: 2,360
Searched 8,000 files | Candidates: 2,523
Searched 8,500 files | Candidates: 2,675
Searched 9,000 files | Candidates: 2,820
Searched 9,500 files | Candidates: 2,965
Searched 10,000 files | Candidates: 3,130
Searched 10,500 files | Candidates: 3,276
Searched 11,000 files | Candidates: 3,428
Searched 11,500 files | Candidates: 3,585
Searched 12,000 files | Candidates: 3,719
Searched 12,500 files | C

In [17]:
import json

with output_file.open("r", encoding="utf-8") as f:
    candidate = json.loads(next(f))

print("FILE:", candidate["title"])

texts = candidate.get("RevX_OCR_text") or []
links = candidate.get("pageLink") or []
numbers = candidate.get("pageNumber") or []

shown = 0

for i, text in enumerate(texts):
    if not isinstance(text, str):
        continue

    normalized = " ".join(text.lower().split())

    if "widow of" in normalized:
        print("\nPAGE:", numbers[i] if i < len(numbers) else i + 1)
        print("SOURCE:", links[i] if i < len(links) else "")
        print(text)

        shown += 1
        if shown == 3:
            break

FILE: Revolutionary War Pension and Bounty Land Warrant Application File W. 26,542, John Torrey, Conn.

PAGE: 2
SOURCE: https://catalog.archives.gov/id/111403885?objectPage=2
Reported
4683.
New York Albany
Abigail Torrey widow of John Torrey Con who was a Private in the Revolution
Inscribed on the Roll at the rate of 27 Dollars 22 Cents per annum, to commence on the 4th day of March, 1843.
Certificate of Pension issued the 25th day of Jan'y 1844 and sent to H. U. Dyer Batavia N. York
[Act of March 3, 1843.]
Recorded in Book A Vol. 1. Page 139.

PAGE: 10
SOURCE: https://catalog.archives.gov/id/111403885?objectPage=10
Declaration of Abigail Torrey widow of John Torrey, under act of July 7. 1838.
advised 8 mo 5 ds / mi 27.22
H. W. Loper, Agent Petersburg, N. Y.

PAGE: 11
SOURCE: https://catalog.archives.gov/id/111403885?objectPage=11
State of New York Genesee County }ss. On this 26 th July 1843 personally appeared before the undersigned a Judge of Genesee County Court, Abigail Torrey a re

In [18]:
import json
import pandas as pd
from pathlib import Path

candidate_file = Path(
    "search_results/widow_pension_candidates.jsonl"
)

rows = []

with candidate_file.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        texts = record.get("RevX_OCR_text") or []
        numbers = record.get("pageNumber") or []
        links = record.get("pageLink") or []

        for i, text in enumerate(texts):
            if not isinstance(text, str):
                continue

            normalized = " ".join(text.lower().split())

            if (
                "widow of" in normalized
                and "per annum" in normalized
            ):
                rows.append({
                    "file_id": record["NAID"],
                    "file_title": record["title"],
                    "page_number": (
                        numbers[i] if i < len(numbers) else None
                    ),
                    "page_url": (
                        links[i] if i < len(links) else None
                    ),
                    "text_original": text,
                    "review_status": "unreviewed"
                })

review_pages = pd.DataFrame(rows)

review_pages.to_csv(
    "search_results/widow_pension_review_pages.csv",
    index=False
)

print("Candidate pages:", len(review_pages))
print("Distinct files:", review_pages["file_id"].nunique())

Candidate pages: 75944
Distinct files: 21717


In [19]:
import json
import pandas as pd
from pathlib import Path

candidate_file = Path(
    "search_results/widow_pension_candidates.jsonl"
)

rows = []

with candidate_file.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        texts = record.get("RevX_OCR_text") or []
        numbers = record.get("pageNumber") or []
        links = record.get("pageLink") or []

        for i, text in enumerate(texts):
            if not isinstance(text, str):
                continue

            normalized = " ".join(text.lower().split())

            if "widow of" in normalized and "per annum" in normalized:
                rows.append({
                    "file_id": record["NAID"],
                    "file_title": record["title"],
                    "page_number": (
                        numbers[i] if i < len(numbers) else None
                    ),
                    "page_url": (
                        links[i] if i < len(links) else None
                    ),
                    "text_original": text,
                    "review_status": "unreviewed"
                })

columns = [
    "file_id", "file_title", "page_number",
    "page_url", "text_original", "review_status"
]

review_pages = pd.DataFrame(rows, columns=columns)

review_pages.to_csv(
    "search_results/widow_pension_review_pages.csv",
    index=False
)

print("Candidate pages:", len(review_pages))
print("Distinct files:", review_pages["file_id"].nunique())
print("Review table saved.")

Candidate pages: 75944
Distinct files: 21717
Review table saved.


In [21]:
examples = review_pages.sample(n=5, random_state=42)

for index, page in examples.iterrows():
    print("\n" + "=" * 60)
    print("ROW:", index)
    print("FILE:", page["file_title"])
    print("PAGE:", page["page_number"])
    print("SOURCE:", page["page_url"])
    print("\n", page["text_original"])


ROW: 18713
FILE: Revolutionary War Pension and Bounty Land Warrant Application File W 2274, for Benjamin Thurber, Massachusetts
PAGE: 11
SOURCE: https://catalog.archives.gov/id/144333983?objectPage=11

 Commonwealth of Massachusetts
County of Bristol.
on this fifth day of April 1853, personally appeared before the Court of Probate within and for said County, Maryann Thurbur now resident in Seekonk in said County aged Eightyfive years who being duly sworn according to law, doth on her oath make the following declaration in order to obtain the benefit of the provisions of the second section of the Act of Congress, passed February 3 d 1853 granting pensions to the widows of the Officers and soldiers of the Revolution. Married since the year, Eighteen hundred, viz. That she is the widow of Benjamin Thurbur late of said Seekonk, deceased, who was a soldier of the Revolution and when the act of Congress of June 7 th 1832 was passed he the said Benjamin Thurbur applied for and obtained a pen

In [22]:
certificate_mask = review_pages["text_original"].str.contains(
    r"certificate\s+of\s+pension|inscribed\s+on\s+the\s+roll",
    case=False,
    na=False,
    regex=True
)

certificate_pages = review_pages[certificate_mask].copy()

print("Possible award pages:", len(certificate_pages))

for _, page in certificate_pages.head(3).iterrows():
    print("\nFILE:", page["file_title"])
    print("PAGE:", page["page_number"])
    print("SOURCE:", page["page_url"])
    print(page["text_original"])

Possible award pages: 40083

FILE: Revolutionary War Pension and Bounty Land Warrant Application File W. 26,542, John Torrey, Conn.
PAGE: 2
SOURCE: https://catalog.archives.gov/id/111403885?objectPage=2
Reported
4683.
New York Albany
Abigail Torrey widow of John Torrey Con who was a Private in the Revolution
Inscribed on the Roll at the rate of 27 Dollars 22 Cents per annum, to commence on the 4th day of March, 1843.
Certificate of Pension issued the 25th day of Jan'y 1844 and sent to H. U. Dyer Batavia N. York
[Act of March 3, 1843.]
Recorded in Book A Vol. 1. Page 139.

FILE: Revolutionary War Pension and Bounty Land Warrant Application File W. 26,542, John Torrey, Conn.
PAGE: 32
SOURCE: https://catalog.archives.gov/id/111403885?objectPage=32
New York 7877
Abigail Torrey
Abigail Torrey widow of John Torrey
who was a pensioner under the Act of and who died on the 9 March, 1821- of Seneca (in the State of N. Yk) who was a private in the company — commanded by Captain Clark of the Regt:

In [39]:
import re
import pandas as pd

def extract_award(text):
    text = " ".join(text.split())

    def find(pattern):
        match = re.search(pattern, text, flags=re.I)
        return match.group(1).strip() if match else None

    # Keep the surrounding name passage instead of guessing name boundaries.
    name_passage = find(
        r"(.{0,100}\bwidow\s+of\b.{0,150})"
    )

    rate_matches = list(re.finditer(
        r"(?:at\s+the\s+|the\s+)?rate\s+of\s+"
        r"(\d[\d,]*(?:\.\d+)?)\s*[-—–]*\s*Dollars"
        r"\s*[-—–]*\s*(\d{1,2})?\s*Cents\s+per\s+annum",
        text,
        flags=re.I
    ))

    annual_rate = None
    rate_passage = None
    flags = []

    if len(rate_matches) == 1:
        match = rate_matches[0]
        dollars = float(match.group(1).replace(",", ""))
        cents = match.group(2)
        rate_passage = match.group(0)

        if cents is not None and int(cents) < 100:
            annual_rate = round(dollars + int(cents) / 100, 2)
        else:
            flags.append("blank_or_unclear_cents")
    elif len(rate_matches) > 1:
        flags.append("multiple_rates")
    else:
        flags.append("rate_not_parsed")

    if re.search(r"cancel|supersed|new\s+certificate", text, re.I):
        flags.append("possible_cancellation_or_replacement")

    if re.search(r"administrator|adm\b|children|heirs", text, re.I):
        flags.append("possible_heirs_or_administrator")

    return {
        "name_passage": name_passage,
        "annual_rate_draft": annual_rate,
        "rate_passage": rate_passage,
        "rank_draft": find(
            r"who\s+(?:was|served).*?\bas\s+a[n]?\s+"
            r"(Private|Captain|Lieutenant|Sergeant|Corporal|Major|Colonel)\b"
        ),
        "start_date_passage": find(
            r"to\s+commence\s+on\s+(.{0,100})"
        ),
        "certificate_date_passage": find(
            r"certificate\s+of\s+pension\s+issued\s+(.{0,100})"
        ),
        "act_passage": find(r"\[Acts?\s+([^\]]+)\]"),
        "review_flags": "; ".join(flags),
        "extraction_status": "unverified"
    }

extracted = pd.DataFrame(
    certificate_pages["text_original"]
    .apply(extract_award)
    .tolist()
)

awards_draft = pd.concat(
    [
        certificate_pages.reset_index(drop=True),
        extracted
    ],
    axis=1
)

awards_draft.to_csv(
    "search_results/widow_awards_draft_v01.csv",
    index=False
)

print("Pages processed:", len(awards_draft))
print(
    "Numeric annual rates extracted:",
    awards_draft["annual_rate_draft"].notna().sum()
)

display(
    awards_draft[
        ["name_passage", "annual_rate_draft", "review_flags"]
    ].head(10)
)

Pages processed: 40083
Numeric annual rates extracted: 15286


,name_passage,annual_rate_draft,review_flags
0,Reported 4683. New York Albany Abigail Torrey ...,27.22,
1,New York 7877 Abigail Torrey Abigail Torrey wi...,27.22,
2,3332 A New York. Sarah Torry widow of Amos Tor...,26.66,possible_cancellation_or_replacement
3,10836. A. New York. Sarah Torry Widow of Amos ...,26.66,
4,Republished 5335. 104. New York. Sarah Torry w...,51.21,
5,8752 A. New York. Sarah Torry widow of Amos To...,26.66,
6,10836. A. New York. Sarah Torry Widow of Amos ...,51.21,
7,8752. A. New York Sarah Torry widow of Amos To...,51.21,
8,New York 3738 Abigail Torrey widow of Jonathan...,NaN,rate_not_parsed
9,"ril 1838, from 4 Sep 1839 to 4 th March 1840. ...",NaN,rate_not_parsed


In [35]:
print(
    awards_draft["review_flags"]
    .replace("", "no_flags")
    .value_counts()
    .to_string()
)

review_flags
no_flags                                                                                         14510
blank_or_unclear_cents                                                                           13694
rate_not_parsed                                                                                  10171
possible_heirs_or_administrator                                                                    559
rate_not_parsed; possible_heirs_or_administrator                                                   406
blank_or_unclear_cents; possible_heirs_or_administrator                                            309
rate_not_parsed; possible_cancellation_or_replacement                                              150
blank_or_unclear_cents; possible_cancellation_or_replacement                                       122
possible_cancellation_or_replacement                                                               102
possible_cancellation_or_replacement; possible_heirs_or_admi

In [36]:
blank_cents = awards_draft[
    awards_draft["review_flags"].str.contains(
        "blank_or_unclear_cents", na=False
    )
]

for _, row in blank_cents.head(3).iterrows():
    print("\nSOURCE:", row["page_url"])
    print("RATE:", row["rate_passage"])


SOURCE: https://catalog.archives.gov/id/111404196?objectPage=3
RATE: at the rate of 80 Dollars Cents per annum

SOURCE: https://catalog.archives.gov/id/111404196?objectPage=18
RATE: at the rate of 80 Dollars Cents per annum

SOURCE: https://catalog.archives.gov/id/111404196?objectPage=33
RATE: at the rate of 80 Dollars — Cents per annum


In [41]:
rates = awards_draft["rate_passage"].str.extract(
    r"(?i)rate\s+of\s+"
    r"(?P<dollars>\d[\d,]*(?:\.\d+)?)"
    r"\s*[-—–]*\s*Dollars"
    r"\s*[-—–]*\s*(?P<cents>\d{1,2})?\s*Cents"
)

awards_draft["dollars_stated"] = pd.to_numeric(
    rates["dollars"].str.replace(",", "", regex=False),
    errors="coerce"
)

awards_draft["cents_stated"] = pd.to_numeric(
    rates["cents"],
    errors="coerce"
).astype("Int64")

awards_draft.to_csv(
    "search_results/widow_awards_draft_v02.csv",
    index=False
)

print(
    "Pages with stated dollar amounts:",
    awards_draft["dollars_stated"].notna().sum()
)

Pages with stated dollar amounts: 29552


In [38]:
missed = awards_draft[
    awards_draft["review_flags"].str.contains(
        "rate_not_parsed", na=False
    )
]

examples = missed.sample(
    n=min(5, len(missed)),
    random_state=42
)

for _, row in examples.iterrows():
    text = " ".join(row["text_original"].split())
    match = re.search(
        r".{0,180}\bper\s+annum\b.{0,80}",
        text,
        flags=re.I
    )

    print("\nSOURCE:", row["page_url"])
    print(match.group(0) if match else text[:500])


SOURCE: https://catalog.archives.gov/id/54084901?objectPage=13
New-York - Albany Heedah Brownson born widow of Isaac Brownson who was a Private in the Connecticut Line for two years. Inscribed on the Roll at the rate of Eighty Dollars Cents per annum, to commence on the 4th day of March, 1843. Certificate of Pension issued the 1

SOURCE: https://catalog.archives.gov/id/53895102?objectPage=17
setts 2107 Annas Barnard widow of Jonathan Barnard who was a Private in the Massachusetts line for 2 years Dept Nov 4 Inscribed on the Roll at the rate of Eighty — Dollars — Cents per annum, to commence on the 4th day of March, 1843. Certificate of Pension issued the f

SOURCE: https://catalog.archives.gov/id/196315261?objectPage=13
the State of Ct. who was a private in the Co. commanded by Captain Riley — of the Regt. commanded by Col. Sage — in the Connecticut line for militia for 16 wks. increased from $30 per annum increased from $30 to $53.33 per annum - Letter to S. H. Parsons 7. June 1843. 

In [42]:
missed = awards_draft[
    awards_draft["review_flags"].str.contains(
        "rate_not_parsed", na=False
    )
]

for text in missed["text_original"].head(5):
    text = " ".join(text.split())
    match = re.search(
        r"(?:at\s+the\s+|the\s+)?rate\s+of\s+.{0,150}",
        text,
        flags=re.I
    )
    print(match.group(0) if match else "Different format")
    print()

at the rate of Fifty — Dollars — Cents per annum, to commence on the 4th day of March, 1843. Certificate of Pension issued the twelfth day of December 1843 and sent 

at the rate of 40 Dollars per annum, to commence on the 4 th day of March, 1836. 14. June 1837 Certificate of Pension issued the 3 rd day of May 1839, and sent to Ma

at the rate of Eighty Dollars per annum, until the termination of the period for which the said certificate was granted, namely the 4 th day of March 1845, when it w

Different format

Different format



In [43]:
number_words = {
    "zero": 0, "one": 1, "two": 2, "three": 3,
    "four": 4, "five": 5, "six": 6, "seven": 7,
    "eight": 8, "nine": 9, "ten": 10,
    "eleven": 11, "twelve": 12, "thirteen": 13,
    "fourteen": 14, "fifteen": 15, "sixteen": 16,
    "seventeen": 17, "eighteen": 18, "nineteen": 19,
    "twenty": 20, "thirty": 30, "forty": 40,
    "fifty": 50, "sixty": 60, "seventy": 70,
    "eighty": 80, "ninety": 90
}

def parse_number(value):
    value = value.lower().strip()

    if re.fullmatch(r"\d+(?:,\d{3})*(?:\.\d+)?", value):
        return float(value.replace(",", ""))

    words = re.sub(r"[-—–]", " ", value).split()
    total = 0

    for word in words:
        if word == "and":
            continue
        if word in number_words:
            total += number_words[word]
        elif word == "hundred" and total > 0:
            total *= 100
        else:
            return None

    return float(total) if words else None

def extract_stated_dollars(text):
    text = " ".join(text.split())

    matches = list(re.finditer(
        r"\brate\s+of\s+"
        r"([A-Za-z0-9,.\s—–-]{1,90}?)\s+Dollars\b"
        r"(?:\s*[-—–]*\s*"
        r"(?:[A-Za-z0-9\s—–-]{0,60}?\s*)?Cents\b)?"
        r"\s+per\s+annum\b",
        text,
        flags=re.I
    ))

    if len(matches) != 1:
        return None, None

    match = matches[0]
    value = parse_number(match.group(1).strip(" -—–"))

    return value, match.group(0)

results = awards_draft["text_original"].apply(
    extract_stated_dollars
)

added = 0

for index, (dollars, passage) in results.items():
    if (
        pd.isna(awards_draft.at[index, "dollars_stated"])
        and dollars is not None
    ):
        awards_draft.at[index, "dollars_stated"] = dollars
        awards_draft.at[index, "rate_passage"] = passage
        added += 1

awards_draft.to_csv(
    "search_results/widow_awards_draft_v03.csv",
    index=False
)

print("Additional stated dollar amounts:", added)
print(
    "Total pages with stated dollar amounts:",
    awards_draft["dollars_stated"].notna().sum()
)

Additional stated dollar amounts: 2156
Total pages with stated dollar amounts: 31708


In [44]:
newly_parsed = awards_draft[
    awards_draft["annual_rate_draft"].isna()
    & awards_draft["dollars_stated"].notna()
    & awards_draft["review_flags"].str.contains(
        "rate_not_parsed", na=False
    )
]

display(
    newly_parsed[
        ["dollars_stated", "rate_passage", "page_url"]
    ].sample(n=10, random_state=42)
)

,dollars_stated,rate_passage,page_url
11156,40.0,rate of 40 Dollars in Cents per annum,https://catalog.archives.gov/id/196129502?obje...
26256,74.0,rate of 74 - Dollars 06 - Cents per annum,https://catalog.archives.gov/id/54131217?objec...
3145,80.0,rate of Eighty Dollars per annum,https://catalog.archives.gov/id/111687522?obje...
5552,80.0,rate of 80 Dollars and Cents per annum,https://catalog.archives.gov/id/144065224?obje...
17899,80.0,rate of 80 Dollars and Cents per annum,https://catalog.archives.gov/id/196619746?obje...
9381,38.0,rate of 38 - Dollars 33 - Cents per annum,https://catalog.archives.gov/id/144302001?obje...
4557,70.0,rate of 70 — Dollars 87 — Cents per annum,https://catalog.archives.gov/id/111771000?obje...
4318,60.0,rate of sixty dollars per annum,https://catalog.archives.gov/id/111756854?obje...
14118,80.0,rate of Eighty Dollars Cents per annum,https://catalog.archives.gov/id/196293820?obje...
8436,61.0,rate of 61 — Dollars 10 — Cents per annum,https://catalog.archives.gov/id/144249989?obje...


In [45]:
# Recover numeric cents, allowing dashes around the number.
cents_text = awards_draft["rate_passage"].str.extract(
    r"(?i)Dollars\s*[-—–]*\s*(\d{1,2})"
    r"\s*[-—–]*\s*Cents",
    expand=False
)

cents = pd.to_numeric(cents_text, errors="coerce")

mask = (
    awards_draft["annual_rate_draft"].isna()
    & awards_draft["dollars_stated"].notna()
    & cents.notna()
)

awards_draft.loc[mask, "cents_stated"] = cents[mask].astype("Int64")

awards_draft.loc[mask, "annual_rate_draft"] = (
    awards_draft.loc[mask, "dollars_stated"]
    + cents[mask] / 100
).round(2)

awards_draft.to_csv(
    "search_results/widow_awards_draft_v04.csv",
    index=False
)

print("Additional complete annual rates:", mask.sum())
print(
    "Total complete annual rates:",
    awards_draft["annual_rate_draft"].notna().sum()
)

Additional complete annual rates: 729
Total complete annual rates: 16015


In [46]:
print(
    awards_draft["rank_draft"]
    .fillna("Not extracted")
    .value_counts()
    .to_string()
)

rank_draft
Not extracted    30708
Private           5902
private           2841
Sergeant           234
Lieutenant         125
Corporal           120
Captain             93
Major               37
Colonel             10
sergeant             8
corporal             5


In [47]:
awards_draft["rank_draft"] = (
    awards_draft["rank_draft"].str.title()
)

missing_rank = awards_draft[
    awards_draft["rank_draft"].isna()
]

for text in missing_rank["text_original"].head(5):
    text = " ".join(text.split())

    match = re.search(
        r".{0,60}\b"
        r"(?:private|captain|lieutenant|sergeant|corporal|major|colonel)"
        r"\b.{0,60}",
        text,
        flags=re.I
    )

    print(match.group(0) if match else "No full rank word found")
    print()

rk Albany Abigail Torrey widow of John Torrey Con who was a Private in the Revolution Inscribed on the Roll at the rate of 27 D

 9 March, 1821- of Seneca (in the State of N. Yk) who was a private in the company — commanded by Captain Clark of the Regt: co

5 August 1831 of Tompkins in the State of N. York who was a Private in the Rev. Comp y commanded by Captain Osborne Reg t comma

8752 A. New York. Sarah Torry widow of Amos Torry who was a Private in the Revolution for 8 months Inscribed on the Roll at the

h Aug t 1831 of Tipton Co in the State of N. York who was a Private in the Company commanded by Captain Asburne of the Regt com



In [48]:
rank_pattern = (
    r"\bwho\s+(?:was|served)\s+"
    r"(?:(?:as\s+)?(?:a|an)\s+)?"
    r"(Private|Captain|Lieutenant|Sergeant|Corporal|Major|Colonel)\b"
)

rank_found = (
    awards_draft["text_original"]
    .str.replace(r"\s+", " ", regex=True)
    .str.extract(rank_pattern, flags=re.I, expand=False)
    .str.title()
)

mask = awards_draft["rank_draft"].isna() & rank_found.notna()

awards_draft.loc[mask, "rank_draft"] = rank_found[mask]

awards_draft.to_csv(
    "search_results/widow_awards_draft_v05.csv",
    index=False
)

print("Additional ranks extracted:", mask.sum())
print(
    awards_draft["rank_draft"]
    .fillna("Not extracted")
    .value_counts()
    .to_string()
)

Additional ranks extracted: 19209
rank_draft
Private          26065
Not extracted    11499
Sergeant           984
Captain            469
Corporal           469
Lieutenant         411
Major              142
Colonel             44


In [49]:
print(
    awards_draft["act_passage"]
    .fillna("Not extracted")
    .value_counts()
    .head(20)
    .to_string()
)

act_passage
Not extracted                           24513
of March 3, 1843.                        7456
July 7, 1838.                            5061
of March 3, 1843, & June 17, 1844.       1304
of March 3, 1843, and June 17, 1844.      737
of JUNE 17, 1844.                         664
of March 3, 1843                          105
July 7, 1838                               57
July 7, 1835.                              14
of JUNE 17, 1844                            9
of March 3, 1843, and June 17, 1844         7
July 7, 1833.                               5
of March 3, 1842                            5
July 7 1838.                                5
of March 3, 1843,                           5
of June 17, 1844.                           5
of March 3, 1843; and June 17, 1844.        4
July 7, 1836                                4
July 7, 1832                                4
of March 3, 1845                            3


In [50]:
missing_act = awards_draft[
    awards_draft["act_passage"].isna()
]

examples = missing_act.sample(n=5, random_state=42)

for _, row in examples.iterrows():
    text = " ".join(row["text_original"].split())

    passages = re.findall(
        r"\bacts?\b.{0,140}",
        text,
        flags=re.I
    )

    print("\nSOURCE:", row["page_url"])
    print("\n".join(passages) if passages else "No act wording found")


SOURCE: https://catalog.archives.gov/id/54955691?objectPage=21
Act of 1832 and who died on the 20 th August 1836 of Mercer Co. in the State of New Jersey who was a Private in the Comp y commanded by Captain
Act July 7, } 1838. } Recorded by W. H. Addison Clerk, Book A Vol. 2 nd Page 125

SOURCE: https://catalog.archives.gov/id/111498104?objectPage=2
act February 3, 1853, Page 131 Vol. A 1828

SOURCE: https://catalog.archives.gov/id/54157849?objectPage=25
act February 2, 1848, Page 336 Vol. 4

SOURCE: https://catalog.archives.gov/id/196103493?objectPage=3
act February 2, 1848, Page 21 Vol. 2

SOURCE: https://catalog.archives.gov/id/54020917?objectPage=3
act February 2, 1848, Page 333 Vol. 4


In [51]:
def collect_act_references(text):
    text = " ".join(text.split())

    references = re.findall(
        r"\bacts?\b.{0,160}",
        text,
        flags=re.I
    )

    return " | ".join(references) if references else None

awards_draft["act_references_raw"] = (
    awards_draft["text_original"].apply(collect_act_references)
)

awards_draft.to_csv(
    "search_results/widow_awards_draft_v06.csv",
    index=False
)

print(
    "Pages with act references:",
    awards_draft["act_references_raw"].notna().sum()
)

display(
    awards_draft[
        ["act_passage", "act_references_raw"]
    ].sample(n=5, random_state=42)
)

Pages with act references: 38937


,act_passage,act_references_raw
37615,"of March 3, 1843.","Act of March 3, 1843.] Recorded in Book A Vol...."
36481,"of March 3, 1843.","Act of March 3, 1843.] Recorded in Book A Vol...."
17867,"of March 3, 1843, & June 17, 1844.","Acts of March 3, 1843, & June 17, 1844.] Recor..."
14307,"July 7, 1838.",Act of 1832 and who died on the 16. Novem r 40...
25736,NaN,"act February 2, 1848, Page 38 Vol. 3,"


In [52]:
def award_act_context(text):
    text = " ".join(text.split())

    certificate = re.search(
        r"certificate\s+of\s+pension\s+issued",
        text,
        flags=re.I
    )

    if certificate is None:
        return None

    remaining_text = text[certificate.end():]

    act = re.search(
        r"\bacts?\b.{0,160}",
        remaining_text,
        flags=re.I
    )

    return act.group(0) if act else None

awards_draft["award_act_context_draft"] = (
    awards_draft["text_original"].apply(award_act_context)
)

awards_draft.to_csv(
    "search_results/widow_awards_draft_v07.csv",
    index=False
)

print(
    "Pages with act references after certificate wording:",
    awards_draft["award_act_context_draft"].notna().sum()
)

for text in (
    awards_draft["award_act_context_draft"]
    .dropna()
    .sample(n=5, random_state=42)
):
    print("\n", text)

Pages with act references after certificate wording: 38065

 Act July 7, } { 1838. } Recorded by R. Burdine Clerk, Book A Vol. 1 - Page 38

 act February 8, 1858, Page 95 Vol. C

 act February 2. 1848, Page 27 Vol. 2,

 act February 2, 1848, Page 346, Vol. 3. Let Hon J. Shielas Apr 26. 1851— A. H. Mackland Jany 6 1855

 Act July 7, 1838.] Recorded by J. H. Swann Clerk, Book D Vol. 1 Page 263


In [53]:
month_pattern = (
    r"January|February|March|April|May|June|July|"
    r"August|September|October|November|December"
)

def extract_act_dates(context):
    if not isinstance(context, str):
        return None

    dates = re.findall(
        rf"\b({month_pattern})\s+"
        r"(\d{1,2})"
        r"(?:st|nd|rd|th)?"
        r"[\s,.;:{}\[\]—–-]+"
        r"(\d{4})\b",
        context,
        flags=re.I
    )

    return (
        " | ".join(
            f"{month.title()} {int(day)}, {year}"
            for month, day, year in dates
        )
        if dates else None
    )

awards_draft["award_act_dates_draft"] = (
    awards_draft["award_act_context_draft"]
    .apply(extract_act_dates)
)

print(
    awards_draft["award_act_dates_draft"]
    .fillna("Date not parsed")
    .value_counts()
    .head(25)
    .to_string()
)

awards_draft.to_csv(
    "search_results/widow_awards_draft_v08.csv",
    index=False
)

award_act_dates_draft
July 7, 1838                        11718
March 3, 1843                        6604
February 2, 1848                     5994
July 4, 1836                         3696
Date not parsed                      2884
March 3, 1843 | June 17, 1844        2729
February 3, 1853                     2261
June 17, 1844                         673
July 29, 1848                         580
June 7, 1832                          358
February 9, 1848                      154
July 20, 1848                         118
February 29, 1848                      87
February 2, 1818                       62
January 2, 1848                        50
February 3, 1848                       48
July 29, 1818                          38
March 3, 1843 | June 17, 1846          36
February 8, 1853                       31
February 3, 1853 | July 27, 1868       29
January 3, 1853                        25
July 20, 1818                          24
March 3, 1843 | June 17, 1843          23
July 22, 184

In [54]:
check = awards_draft[
    awards_draft["award_act_dates_draft"].eq("February 9, 1848")
].iloc[0]

print("SOURCE:", check["page_url"])
print("ACT CONTEXT:", check["award_act_context_draft"])
print("\nFULL TEXT:\n", check["text_original"])

SOURCE: https://catalog.archives.gov/id/111408757?objectPage=3
ACT CONTEXT: act February 9, 1848, Page 256 Vol. 4, 3/53 7 July 22/68 S. M. Smith Sent

FULL TEXT:
 release
No. 4,872
Tennessee (Knoxville)
Elizabeth M. Townsend
Widow of George Townsend who served in the Revolutionary
war, as a Private
Inscribed on the Roll at the rate of 96 Dollars 00
Cents per annum, to commence on the 4th day of March, 1848.
27 th July 1868. Wd. Pts. Mad. since Certificate of Pension issued the 13 th day of March 1869 and sent to Tucker & Selly
Present
Recorded on Roll of Pensioners under act February 9, 1848, Page 256 Vol. 4,
3/53 7 July 22/68
S. M. Smith
Sent


In [55]:
mask = (
    awards_draft["file_id"].astype(str).eq("111408757")
    & awards_draft["page_number"].astype(str).eq("3")
)

awards_draft.loc[mask, "image_review_notes"] = (
    "Scan: printed March 4, 1848 start date is crossed out; "
    "handwritten date appears to be July 27, 1868. "
    "Printed act date appears to be February 2, 1848; OCR says 9. "
    "Do not use the superseded printed start date to calculate a gap."
)

awards_draft.to_csv(
    "search_results/widow_awards_draft_v09.csv",
    index=False
)

print("Rows annotated:", mask.sum())

Rows annotated: 1


In [56]:
validation_sample = awards_draft.sample(
    n=20,
    random_state=2026
).copy()

validation_sample["amount_matches_scan"] = ""
validation_sample["rank_matches_scan"] = ""
validation_sample["act_matches_scan"] = ""
validation_sample["widow_award_confirmed"] = ""
validation_sample["validation_notes"] = ""

validation_sample.to_csv(
    "search_results/validation_sample_20.csv",
    index=False
)

print("Saved 20-page validation sample.")

Saved 20-page validation sample.


In [58]:
mask = (
    awards_draft["file_id"].astype(str).eq("54238880")
    & pd.to_numeric(
        awards_draft["page_number"], errors="coerce"
    ).eq(3)
)

awards_draft.loc[mask, "award_act_date_reviewed"] = "1848-02-02"
awards_draft.loc[mask, "image_review_notes"] = (
    "Scan confirms 40 dollars, blank cents, and Private. "
    "Printed act month/day crossed out and replaced with "
    "'2d Feby'; printed year 1848 retained. "
    "OCR July 29, 1848 does not reflect the revision. "
    "Name not checked in supplied screenshots."
)

awards_draft.to_csv(
    "search_results/widow_awards_draft_v10.csv",
    index=False
)

print("Rows annotated:", mask.sum())

Rows annotated: 1


In [59]:
review = validation_sample.iloc[1]

print("SOURCE:", review["page_url"])
print("DOLLARS:", review["dollars_stated"])
print("CENTS:", review["cents_stated"])
print("RANK:", review["rank_draft"])
print("ACT:", review["award_act_dates_draft"])
print("\nOCR TEXT:\n", review["text_original"])

SOURCE: https://catalog.archives.gov/id/53970567?objectPage=23
DOLLARS: nan
CENTS: <NA>
RANK: nan
ACT: June 17, 1844

OCR TEXT:
 New York. 254.
Anne Elizabeth Bircki widow of Peter Bircki who was a Pri & Batteausman in the New York Line.
Husb. died 29 th June 1843.
Inscribed on the Roll at the rate of 56 ~ Dollars 30 ~ Cents per annum, to commence on the 4th day of March, 1844. 29 th June 1845.
Certificate of Pension issued the 15 th day of Sept. 1846. and sent to
Hiram Hubbell, Richland, N. York.
[Act of JUNE 17, 1844.]
Recorded in Book A.
Vol. 2. Page 72.
D. H. Wood & Co.
Rep d Lt. G. J. W. Munger August 21 st '47


In [60]:
mask = (
    awards_draft["file_id"].astype(str).eq("53970567")
    & pd.to_numeric(
        awards_draft["page_number"], errors="coerce"
    ).eq(23)
)

awards_draft.loc[mask, "annual_rate_reviewed"] = 56.30
awards_draft.loc[mask, "start_date_reviewed"] = "1845-06-29"
awards_draft.loc[mask, "award_act_date_reviewed"] = "1844-06-17"
awards_draft.loc[mask, "service_description_reviewed"] = (
    "Pri & Batteausman"
)
awards_draft.loc[mask, "image_review_notes"] = (
    "Scan shows 56 dollars 30 cents. "
    "Printed March 4, 1844 start date crossed out; "
    "replaced with June 29, 1845. "
    "Husband's death year appears to be 1845; "
    "needs closer verification against OCR's 1843."
)

awards_draft.to_csv(
    "search_results/widow_awards_draft_v11.csv",
    index=False
)

print("Rows annotated:", mask.sum())

Rows annotated: 1


In [61]:
page = validation_sample.iloc[2]

print("SOURCE:", page["page_url"])
print("DOLLARS:", page["dollars_stated"])
print("CENTS:", page["cents_stated"])
print("RANK:", page["rank_draft"])
print("ACT:", page["award_act_dates_draft"])
print("\nOCR TEXT:\n", page["text_original"])

SOURCE: https://catalog.archives.gov/id/111482138?objectPage=13
DOLLARS: 62.0
CENTS: 50
RANK: nan
ACT: July 7, 1838

OCR TEXT:
 DA 17
Libt Payt May 23. 1839
Bellows Falls Vermont
Elizabeth Waldron
widow of John Waldron
who was a pensioner under the Act of and who died on the 29. September 1818 of Windsor Co. in the State of Vt. who was a Discharge in the Comp y commanded by Captain _____ of the Reg t commanded by Col l Elliott in the R. I. d line for 15 mos Private
Inscribed on the Roll of B. Falls Vt. at the rate of 62 Dollars 50 Cents per annum, to commence on the 4th day of March, 1836.
Certificate of Pension issued the 19 th day of Ap l . 1839 and sent Mr. Palmer Woodstock Vt.
Arrears to the 4th of Mar. 39, $187.50 Semi-annual allowance ending Sept. 1, 31.25 $218.75
{ Act July 7, } 1838. }
Recorded by N. Addison Clerk, Book A Vol. 2 d Page 61


In [64]:
import re

rank_words = (
    r"Private|Captain|Lieutenant|Sergeant|"
    r"Corporal|Major|Colonel"
)

for example_number, (_, page) in enumerate(
    validation_sample.iterrows()
):
    if pd.notna(page["rank_draft"]):
        continue

    text = " ".join(page["text_original"].split())

    # Limit the search to text before the pension amount.
    service_text = re.split(
        r"\binscribed\s+on\s+the\s+roll\b",
        text,
        maxsplit=1,
        flags=re.I
    )[0]

    matches = list(re.finditer(
        rf"\b(?:{rank_words})\b",
        service_text,
        flags=re.I
    ))

    if matches:
        print("\n" + "=" * 60)
        print("EXAMPLE:", example_number)
        print("SOURCE:", page["page_url"])
        print("EXTRACTED RANK: Missing")

        for match in matches:
            start = max(0, match.start() - 70)
            end = min(len(service_text), match.end() + 70)
            print("CONTEXT:", service_text[start:end])


EXAMPLE: 2
SOURCE: https://catalog.archives.gov/id/111482138?objectPage=13
EXTRACTED RANK: Missing
CONTEXT: o. in the State of Vt. who was a Discharge in the Comp y commanded by Captain _____ of the Reg t commanded by Col l Elliott in the R. I. d line for
CONTEXT: f the Reg t commanded by Col l Elliott in the R. I. d line for 15 mos Private 

EXAMPLE: 3
SOURCE: https://catalog.archives.gov/id/144050320?objectPage=24
EXTRACTED RANK: Missing
CONTEXT:  the State of Conn. who was a S t M t H t in the Company commanded by Captain _____ of the Regt: commanded by Col Shelton in the Conn line for 12:2

EXAMPLE: 4
SOURCE: https://catalog.archives.gov/id/54509960?objectPage=20
EXTRACTED RANK: Missing
CONTEXT: nnsylvania Phila Rebecca Ellis widow of Stephen. Con n who was a Sp l Sergeant in the Revolution 

EXAMPLE: 10
SOURCE: https://catalog.archives.gov/id/196319850?objectPage=26
EXTRACTED RANK: Missing
CONTEXT: enton in the State of Tenn: who was a priv Wgt: - in the commanded by Captain ____

In [65]:
mask = (
    awards_draft["file_id"].astype(str).eq("54509960")
    & pd.to_numeric(
        awards_draft["page_number"], errors="coerce"
    ).eq(20)
)

awards_draft.loc[mask, "rank_reviewed"] = "Sergeant"
awards_draft.loc[mask, "image_review_notes"] = (
    "Scan supports Sergeant. Preceding abbreviation "
    "is unclear; full designation not resolved. "
    "Printed commencement date is crossed out."
)

awards_draft.to_csv(
    "search_results/widow_awards_draft_v13.csv",
    index=False
)

print("Rows annotated:", mask.sum())

Rows annotated: 1


In [66]:
mask = (
    awards_draft["file_id"].astype(str).eq("196319850")
    & pd.to_numeric(
        awards_draft["page_number"], errors="coerce"
    ).eq(26)
)

awards_draft.loc[mask, "rank_reviewed"] = "Private | Sergeant"
awards_draft.loc[mask, "image_review_notes"] = (
    "Scan appears to list service as both Private and "
    "Sergeant, with separate durations. Exact durations "
    "remain unverified. Do not classify as Private only."
)

awards_draft.to_csv(
    "search_results/widow_awards_draft_v14.csv",
    index=False
)

print("Rows annotated:", mask.sum())

Rows annotated: 1


In [67]:
reviewed_columns = [
    "annual_rate_reviewed",
    "rank_reviewed",
    "award_act_date_reviewed",
    "start_date_reviewed",
    "image_review_notes"
]

reviewed_columns = [
    col for col in reviewed_columns
    if col in awards_draft.columns
]

reviewed_rows = awards_draft.loc[
    awards_draft["image_review_notes"].notna(),
    ["file_id", "page_number", "page_url"] + reviewed_columns
]

display(reviewed_rows)

reviewed_rows.to_csv(
    "search_results/image_reviewed_pages.csv",
    index=False
)

print("Pages with review notes:", len(reviewed_rows))

,file_id,page_number,page_url,annual_rate_reviewed,rank_reviewed,award_act_date_reviewed,start_date_reviewed,image_review_notes
116,111408757,3,https://catalog.archives.gov/id/111408757?obje...,NaN,NaN,NaN,NaN,"Scan: printed March 4, 1848 start date is cros..."
14566,196319850,26,https://catalog.archives.gov/id/196319850?obje...,NaN,Private | Sergeant,NaN,NaN,Scan appears to list service as both Private a...
23541,53970567,23,https://catalog.archives.gov/id/53970567?objec...,56.3,NaN,1844-06-17,1845-06-29,Scan shows 56 dollars 30 cents. Printed March ...
28048,54238880,3,https://catalog.archives.gov/id/54238880?objec...,NaN,NaN,1848-02-02,NaN,"Scan confirms 40 dollars, blank cents, and Pri..."
32533,54509960,20,https://catalog.archives.gov/id/54509960?objec...,NaN,Sergeant,NaN,NaN,Scan supports Sergeant. Preceding abbreviation...


Pages with review notes: 5


In [68]:
mask = (
    awards_draft["file_id"].astype(str).eq("111482138")
    & pd.to_numeric(
        awards_draft["page_number"], errors="coerce"
    ).eq(13)
)

awards_draft.loc[mask, "annual_rate_reviewed"] = 62.50
awards_draft.loc[mask, "rank_reviewed"] = "Private"
awards_draft.loc[mask, "award_act_date_reviewed"] = "1838-07-07"
awards_draft.loc[mask, "image_review_notes"] = (
    "Scan confirms 62 dollars 50 cents, Private, "
    "and Act July 7, 1838. Rank appears after "
    "the service duration and was missed by the parser."
)

awards_draft.to_csv(
    "search_results/widow_awards_draft_v15.csv",
    index=False
)

print("Waldron rows updated:", mask.sum())

Waldron rows updated: 1


In [70]:
review_columns = [
    col for col in awards_draft.columns
    if col.endswith("_reviewed") or col == "image_review_notes"
]

# Match by the original row index retained when we sampled.
for col in review_columns:
    validation_sample[col] = awards_draft[col].reindex(
        validation_sample.index
    )

validation_sample.to_csv(
    "search_results/validation_sample_20.csv",
    index=False
)

print(
    "Sample pages with review notes:",
    validation_sample["image_review_notes"].notna().sum()
)

Sample pages with review notes: 5


In [71]:
page = validation_sample.iloc[3]

print("SOURCE:", page["page_url"])
print("\nOCR TEXT:\n")
print(page["text_original"])

SOURCE: https://catalog.archives.gov/id/144050320?objectPage=24

OCR TEXT:

479
Rept by L Letter to Gen. Ag d May 17 1844
Connecticut
Phoebe James widow of Elizah James who was a pensioner under the Act of 1832 and who died on the 22 Feby 1823 of Rumfield in the State of Conn. who was a S t M t H t in the Company commanded by Captain _____ of the Regt: commanded by Col Shelton in the Conn line for 12:27 p.m. 11. 3 S. 1844
Inscribed on the Roll of Connecticut at the rate of 589. Dollars 17 Cents per annum, to commence on the 4th day of March, 1836.
Increased from $320.00 p. ann. Certificate of Pension issued the 17 day of May 1844 and sent by A. Thompson Ag. Present
Total Amount, $2945.88 [Act July 7, 1838.] Recorded by H. G. Swann Clerk, Book D Vol. 1 Page 85


In [72]:
import re

def rank_check(text):
    text = " ".join(str(text).split())

    # Look only before the award wording.
    service = re.split(
        r"\b(?:inscribed|enrolled)\s+on\s+the\s+roll\b",
        text, maxsplit=1, flags=re.I
    )[0]

    # Remove references to commanding officers.
    service = re.sub(
        r"\bcommanded\s+by\s+"
        r"(?:(?:a|an|the)\s+)?"
        r"(?:Captain|Capt\.?|Colonel|Col\.?|Col\s+l)\b",
        "commanded by [commander]",
        service, flags=re.I
    )

    ranks = re.findall(
        r"\b(?:Private|Priv|Sergeant|Sergt|Captain|"
        r"Lieutenant|Corporal|Major|Colonel)\b",
        service, flags=re.I
    )

    return " | ".join(dict.fromkeys(
        rank.title() for rank in ranks
    ))

awards_draft["rank_mentions_check"] = (
    awards_draft["text_original"].apply(rank_check)
)

possible_misses = awards_draft.loc[
    awards_draft["rank_draft"].isna()
    & awards_draft["rank_mentions_check"].ne("")
].copy()

possible_misses.to_csv(
    "search_results/possible_rank_extraction_misses.csv",
    index=False
)

print("Pages flagged for possible missed ranks:", len(possible_misses))

display(possible_misses[
    ["rank_mentions_check", "page_url"]
].head(10))

Pages flagged for possible missed ranks: 2727


,rank_mentions_check,page_url
19,Sergt,https://catalog.archives.gov/id/111404333?obje...
44,Priv,https://catalog.archives.gov/id/111405378?obje...
47,Lieutenant,https://catalog.archives.gov/id/111405378?obje...
63,Captain | Private | Sergeant | Lieutenant,https://catalog.archives.gov/id/111406157?obje...
95,Private,https://catalog.archives.gov/id/111407885?obje...
100,Private | Corporal,https://catalog.archives.gov/id/111407995?obje...
114,Captain,https://catalog.archives.gov/id/111408589?obje...
141,Sergt | Private,https://catalog.archives.gov/id/111410525?obje...
143,Private | Sergeant,https://catalog.archives.gov/id/111410585?obje...
157,Private,https://catalog.archives.gov/id/111411252?obje...


In [73]:
for _, page in possible_misses.head(5).iterrows():
    text = " ".join(page["text_original"].split())

    service = re.split(
        r"\b(?:inscribed|enrolled)\s+on\s+the\s+roll\b",
        text, maxsplit=1, flags=re.I
    )[0]

    print("\n" + "=" * 60)
    print("SOURCE:", page["page_url"])
    print("RANK WORDS:", page["rank_mentions_check"])
    print("SERVICE TEXT:", service)


SOURCE: https://catalog.archives.gov/id/111404333?objectPage=2
RANK WORDS: Sergt
SERVICE TEXT: Reported Dec 4/48 2299. Massachusetts Elizabeth Torrey widow of Thomas Torrey Jr who served in the Revolutionary war, as a Sergt Mass 

SOURCE: https://catalog.archives.gov/id/111405378?objectPage=65
RANK WORDS: Priv
SERVICE TEXT: New York Sally Torsley, widow of William Torsley, who was a pensioner under the Act of 1818, and who died on the 18 August, 1827, of Orleans (in the State of New-York, who was a priv: & drummer in the _____ commanded by Captain _____ of the Regt - commanded by Col: Warner in the Champlain line for \left. \begin{matrix} 7.13 \text{ days dr: } 1 \\ 13 \text{ priv: } 3 \end{matrix} \right\} Continental & Ft. 

SOURCE: https://catalog.archives.gov/id/111405378?objectPage=76
RANK WORDS: Lieutenant
SERVICE TEXT: New York, Albany Sally Tonsley Widow of William Tonsley who was a pensioner under the Act of 1818 and who died on the 15 th August 1827 of Orleans Co in the Stat

In [74]:
service_pattern = (
    r"\bwho\s+"
    r"(?:was\s+(?:an?\s+)?|"
    r"served\s+in\s+the\s+Revolutionary\s+war"
    r"\s*,?\s*as\s+(?:an?\s+)?)"
    r"(.{1,100}?)"
    r"(?=\s+(?:in\s+the|in\s+"
    r"(?:Mass|Conn|Virginia|New\s+York)\b|"
    r"commanded\b|for\s+\d|"
    r"Inscribed\b)|$)"
)

def get_service_passages(text):
    text = " ".join(str(text).split())

    passages = re.findall(
        service_pattern, text, flags=re.I
    )

    # Exclude phrases describing pensioner status.
    return " | ".join(
        passage.strip()
        for passage in passages
        if not re.search(r"\bpensioner\b", passage, re.I)
    )

awards_draft["service_role_passage_draft"] = (
    awards_draft["text_original"].apply(get_service_passages)
)

display(
    awards_draft.loc[
        possible_misses.head(5).index,
        ["service_role_passage_draft", "page_url"]
    ]
)

,service_role_passage_draft,page_url
19,Sergt Mass,https://catalog.archives.gov/id/111404333?obje...
44,priv: & drummer,https://catalog.archives.gov/id/111405378?obje...
47,1 st Lieutenant,https://catalog.archives.gov/id/111405378?obje...
63,Pri Sgt & Lt,https://catalog.archives.gov/id/111406157?obje...
95,Corpl & Private,https://catalog.archives.gov/id/111407885?obje...


In [77]:
from pathlib import Path

checkpoint = Path("search_results/widow_awards_checkpoint.csv")

awards_draft.to_csv(checkpoint, index=False)

print("Pages saved:", len(awards_draft))
print("Saved to:", checkpoint.resolve())

Pages saved: 40083
Saved to: /Users/catietaras/Desktop/smithsonian_pensions/search_results/widow_awards_checkpoint.csv


In [78]:
# Standardize draft dates; retain multiple acts together.
def format_act_dates(value):
    if not isinstance(value, str):
        return pd.NA

    dates = []
    for part in value.split("|"):
        parsed = pd.to_datetime(part.strip(), errors="coerce")
        if pd.notna(parsed):
            dates.append(parsed.strftime("%Y-%m-%d"))

    return " | ".join(dict.fromkeys(dates)) if dates else pd.NA

draft_acts = (
    awards_draft["award_act_dates_draft"]
    .apply(format_act_dates)
)

reviewed_acts = (
    awards_draft["award_act_date_reviewed"]
    .replace(r"^\s*$", pd.NA, regex=True)
)

awards_draft["award_act_working"] = (
    reviewed_acts.combine_first(draft_acts)
)

awards_draft.to_csv(
    "search_results/widow_awards_checkpoint.csv",
    index=False
)

print(
    awards_draft["award_act_working"]
    .fillna("Unresolved")
    .value_counts()
    .head(15)
    .to_string()
)

award_act_working
1838-07-07                 11719
1843-03-03                  6607
1848-02-02                  5995
1836-07-04                  3697
Unresolved                  2885
1843-03-03 | 1844-06-17     2732
1853-02-03                  2261
1844-06-17                   673
1848-07-29                   579
1832-06-07                   359
1848-02-09                   154
1848-07-20                   118
1848-02-29                    87
1818-02-02                    62
1848-01-02                    50


In [79]:
# Prefer reviewed amounts; otherwise use draft amounts.
awards_draft["annual_rate_working"] = (
    awards_draft["annual_rate_reviewed"]
    .combine_first(awards_draft["annual_rate_draft"])
)

law_summary = (
    awards_draft
    .groupby("award_act_working")
    .agg(
        candidate_pages=("file_id", "size"),
        pages_with_amount=("annual_rate_working", "count"),
        median_annual_amount=("annual_rate_working", "median"),
        lowest_amount=("annual_rate_working", "min"),
        highest_amount=("annual_rate_working", "max")
    )
    .sort_index()
)

display(law_summary.round(2))

,candidate_pages,pages_with_amount,median_annual_amount,lowest_amount,highest_amount
award_act_working,,,,,
1812-02-02,1,0,NaN,NaN,NaN
1813-02-02,1,1,80.00,80.00,80.00
1813-02-21,1,0,NaN,NaN,NaN
1813-03-03 | 1816-06-17,1,0,NaN,NaN,NaN
1813-03-03 | 1840-06-17,1,1,33.33,33.33,33.33
...,...,...,...,...,...
1884-06-19,1,0,NaN,NaN,NaN
1886-07-04,1,1,46.67,46.67,46.67
1888-07-07,1,1,44.44,44.44,44.44


In [80]:
history_columns = [
    "file_id",
    "file_title",
    "page_number",
    "page_url",
    "name_passage",
    "certificate_date_passage",
    "award_act_working",
    "text_original"
]

award_history = awards_draft[history_columns].copy()

award_history["page_sort"] = pd.to_numeric(
    award_history["page_number"], errors="coerce"
)

award_history = (
    award_history
    .sort_values(["file_id", "page_sort"])
    .drop(columns="page_sort")
)

award_history.to_csv(
    "search_results/widow_award_history_draft.csv",
    index=False
)

display(
    award_history[
        ["file_id", "page_number", "certificate_date_passage"]
    ].head(10)
)

,file_id,page_number,certificate_date_passage
0,111403885,2,the 25th day of Jan'y 1844 and sent to H. U. D...
1,111403885,32,the 29 th : — day of November 1843 and sent to...
2,111403929,4,the 9 th day of Feby 1850. and sent to Mrs. R....
3,111403929,55,the 9 th day of Feby 1830 and sent to Hon. R. ...
4,111403929,56,the 16 th day of Feby 1850 and sent to Miss N....
5,111403929,57,the 9th day of July 30 and sent to Mrs. R. L. ...
6,111403929,58,the 16 th day of Febr y . 1850 and sent to Hon...
7,111403929,59,the 16 th day of July 1837 and sent to Rev R. ...
8,111404093,2,the twelfth day of December 1843 and sent to R...
9,111404093,28,"the 3 rd day of May 1839, and sent to Mason Mu..."


In [81]:
award_history["certificate_date_text"] = (
    award_history["certificate_date_passage"]
    .fillna("")
    .str.split(
        r"\band\s+sent\b",
        n=1,
        regex=True
    )
    .str[0]
    .str.strip()
)

# Flag anything without exactly one four-digit year.
award_history["certificate_years_found"] = (
    award_history["certificate_date_text"]
    .str.findall(r"\b(?:17|18|19)\d{2}\b")
)

award_history["certificate_date_needs_review"] = (
    award_history["certificate_years_found"]
    .apply(len)
    .ne(1)
)

display(
    award_history[
        ["file_id", "page_number",
         "certificate_date_text",
         "certificate_date_needs_review"]
    ].head(10)
)

,file_id,page_number,certificate_date_text,certificate_date_needs_review
0,111403885,2,the 25th day of Jan'y 1844,False
1,111403885,32,the 29 th : — day of November 1843,False
2,111403929,4,the 9 th day of Feby 1850.,False
3,111403929,55,the 9 th day of Feby 1830,False
4,111403929,56,the 16 th day of Feby 1850,False
5,111403929,57,the 9th day of July 30,True
6,111403929,58,the 16 th day of Febr y . 1850,False
7,111403929,59,the 16 th day of July 1837,False
8,111404093,2,the twelfth day of December 1843,False
9,111404093,28,"the 3 rd day of May 1839,",False


In [82]:
import re

def parse_certificate_date(value):
    text = value.lower().strip()

    # Normalize common month abbreviations.
    text = re.sub(r"\bjan['’]?\s*y\b", "january", text)
    text = re.sub(r"\bfebr?\s*y\b", "february", text)

    # Handle written ordinal days.
    ordinals = (
        "first second third fourth fifth sixth seventh eighth "
        "ninth tenth eleventh twelfth thirteenth fourteenth "
        "fifteenth sixteenth seventeenth eighteenth nineteenth "
        "twentieth"
    ).split()

    day_words = {word: i + 1 for i, word in enumerate(ordinals)}
    day_words.update({
        "twenty first": 21, "twenty second": 22,
        "twenty third": 23, "twenty fourth": 24,
        "twenty fifth": 25, "twenty sixth": 26,
        "twenty seventh": 27, "twenty eighth": 28,
        "twenty ninth": 29, "thirtieth": 30,
        "thirty first": 31
    })

    text = text.replace("-", " ")
    for word in sorted(day_words, key=len, reverse=True):
        text = re.sub(
            rf"\b{word}\b", str(day_words[word]), text
        )

    text = re.sub(
        r"(\d)\s*(?:st|nd|rd|th)\b", r"\1", text
    )

    match = re.fullmatch(
        r"(?:the\s+)?(\d{1,2})[\s:—–.,]*"
        r"(?:day\s+of\s+)?([a-z]+)[\s.,]+"
        r"((?:17|18|19)\d{2})[\s.,]*",
        text
    )

    if not match:
        return pd.NaT

    day, month, year = match.groups()
    return pd.to_datetime(
        f"{day} {month} {year}",
        errors="coerce",
        dayfirst=True
    )

award_history["certificate_date_draft"] = (
    award_history["certificate_date_text"]
    .apply(parse_certificate_date)
)

print(
    "Dates parsed:",
    award_history["certificate_date_draft"].notna().sum()
)

display(award_history[
    ["file_id", "page_number", "certificate_date_text",
     "certificate_date_draft"]
].head(10))

Dates parsed: 26447


,file_id,page_number,certificate_date_text,certificate_date_draft
0,111403885,2,the 25th day of Jan'y 1844,1844-01-25
1,111403885,32,the 29 th : — day of November 1843,1843-11-29
2,111403929,4,the 9 th day of Feby 1850.,1850-02-09
3,111403929,55,the 9 th day of Feby 1830,1830-02-09
4,111403929,56,the 16 th day of Feby 1850,1850-02-16
5,111403929,57,the 9th day of July 30,NaT
6,111403929,58,the 16 th day of Febr y . 1850,1850-02-16
7,111403929,59,the 16 th day of July 1837,1837-07-16
8,111404093,2,the twelfth day of December 1843,1843-12-12
9,111404093,28,"the 3 rd day of May 1839,",1839-05-03


In [83]:
boundaries = pd.to_datetime([
    "1836-07-04",
    "1838-07-07",
    "1843-03-03",
    "1844-06-17",
    "1848-02-02",
    "1848-07-29",
    "1853-02-03"
])

def certificate_period(date):
    if pd.isna(date):
        return "Date unresolved"

    if date < boundaries[0]:
        return "Before 1836-07-04 — review separately"

    for start, end in zip(boundaries[:-1], boundaries[1:]):
        if start <= date < end:
            last_day = end - pd.Timedelta(days=1)
            return f"{start:%Y-%m-%d} to {last_day:%Y-%m-%d}"

    return "1853-02-03 onward"

award_history["certificate_period_draft"] = (
    award_history["certificate_date_draft"]
    .apply(certificate_period)
)

award_history.to_csv(
    "search_results/widow_award_history_draft.csv",
    index=False
)

print(
    award_history["certificate_period_draft"]
    .value_counts()
    .sort_index()
    .to_string()
)

certificate_period_draft
1836-07-04 to 1838-07-06                  1636
1838-07-07 to 1843-03-02                  3668
1843-03-03 to 1844-06-16                  5904
1844-06-17 to 1848-02-01                  3663
1848-02-02 to 1848-07-28                  1177
1848-07-29 to 1853-02-02                  5682
1853-02-03 onward                         3944
Before 1836-07-04 — review separately      773
Date unresolved                          13636


In [84]:
unresolved_dates = award_history.loc[
    award_history["certificate_date_draft"].isna()
]

print(
    unresolved_dates["certificate_date_text"]
    .replace("", "[No date passage extracted]")
    .value_counts()
    .head(20)
    .to_string()
)

certificate_date_text
[No date passage extracted]           1097
the day of                              15
the 8 th day of Decem r 1843            14
the 3 d day of July 1843                12
the 9 th day of Decem r 1843            12
the 4 th day of Decem r 1843            10
the 22 d day of July 1843               10
the 23 d day of June 1843                9
the 14 th day of De- cember, 1838        9
the 30 th day of Nov r 1843              9
the 11 day of Decr. 1843                 9
the day of Sept. 1848                    8
the 27 th day of November                8
the 7 th day of Decem r 1843             8
the 12 th day of Dec r 1843              8
the 28 th day of Sep- tember, 1838       8
the 30 th day of Aug/53                  7
the 3 rd day of Sept r 39,               7
the 26 th day of Aug/53                  7
the 8 th day of June '39,                7


In [85]:
def normalize_certificate_text(value):
    text = str(value).lower().strip()

    # Rejoin words split by hyphens, such as De- cember.
    text = re.sub(r"(?<=[a-z])-\s+(?=[a-z])", "", text)

    # Normalize abbreviated months.
    month_patterns = {
        "january": r"\bjan(?:uary|['’]?\s*y)?\b",
        "february": r"\bfeb(?:ruary|r\s*y|y)?\b",
        "august": r"\baug(?:ust)?\b",
        "september": r"\bsep(?:tember|t(?:\s*r)?)?\b",
        "october": r"\boct(?:ober|\s*r)?\b",
        "november": r"\bnov(?:ember|\s*r)?\b",
        "december": r"\bdec(?:ember|em\s*r|\s*r)?\b"
    }

    for month, pattern in month_patterns.items():
        text = re.sub(pattern, month, text)

    # Remove ordinal endings and harmless punctuation.
    text = re.sub(
        r"\b(\d{1,2})\s*(?:st|nd|rd|th|d)\b",
        r"\1",
        text
    )
    text = re.sub(r"[.,:—–]", " ", text)

    return " ".join(text.split())

award_history["certificate_date_normalized"] = (
    award_history["certificate_date_text"]
    .apply(normalize_certificate_text)
)

reparsed = (
    award_history["certificate_date_normalized"]
    .apply(parse_certificate_date)
)

fill = (
    award_history["certificate_date_draft"].isna()
    & reparsed.notna()
)

award_history.loc[fill, "certificate_date_draft"] = reparsed[fill]

award_history["certificate_period_draft"] = (
    award_history["certificate_date_draft"]
    .apply(certificate_period)
)

award_history.to_csv(
    "search_results/widow_award_history_draft.csv",
    index=False
)

print("Additional dates parsed:", fill.sum())
print(
    "Dates still unresolved:",
    award_history["certificate_date_draft"].isna().sum()
)

Additional dates parsed: 4164
Dates still unresolved: 9472


In [86]:
file_dates = (
    award_history.groupby("file_id")
    .agg(
        candidate_pages=("page_url", "size"),
        parsed_dates=("certificate_date_draft", "count"),
        earliest_date_found=("certificate_date_draft", "min")
    )
)

file_dates["unresolved_dates"] = (
    file_dates["candidate_pages"] - file_dates["parsed_dates"]
)

file_dates["earliest_date_period_draft"] = (
    file_dates["earliest_date_found"].apply(certificate_period)
)

file_dates.to_csv(
    "search_results/file_date_summary_draft.csv"
)

display(file_dates.head(10))

,candidate_pages,parsed_dates,earliest_date_found,unresolved_dates,earliest_date_period_draft
file_id,,,,,
111403885,2,2,1843-11-29,0,1843-03-03 to 1844-06-16
111403929,6,5,1830-02-09,1,Before 1836-07-04 — review separately
111404093,2,2,1839-05-03,0,1838-07-07 to 1843-03-02
111404196,4,2,1844-08-27,2,1844-06-17 to 1848-02-01
111404239,3,2,1851-12-09,1,1848-07-29 to 1853-02-02
111404292,2,2,1840-04-25,0,1838-07-07 to 1843-03-02
111404333,3,2,1843-11-30,1,1843-03-03 to 1844-06-16
111404357,2,1,1843-06-17,1,1843-03-03 to 1844-06-16
111404415,3,3,1840-08-04,0,1838-07-07 to 1843-03-02


In [87]:
first_files = file_dates.head(3).index

for file_id in first_files:
    print("\nFILE:", file_id)

    pages = award_history[
        award_history["file_id"].eq(file_id)
    ]

    for _, page in pages.iterrows():
        print("PAGE:", page["page_number"])
        print("NAME PASSAGE:", page["name_passage"])
        print("DATE:", page["certificate_date_draft"])
        print()


FILE: 111403885
PAGE: 2
NAME PASSAGE: Reported 4683. New York Albany Abigail Torrey widow of John Torrey Con who was a Private in the Revolution Inscribed on the Roll at the rate of 27 Dollars 22 Cents per annum, to commence on the 4th day of
DATE: 1844-01-25 00:00:00

PAGE: 32
NAME PASSAGE: New York 7877 Abigail Torrey Abigail Torrey widow of John Torrey who was a pensioner under the Act of and who died on the 9 March, 1821- of Seneca (in the State of N. Yk) who was a private in the compan
DATE: 1843-11-29 00:00:00


FILE: 111403929
PAGE: 4
NAME PASSAGE: 3332 A New York. Sarah Torry widow of Amos Torry who served in the Revolutionary war, as a Private for 8 months Inscribed on the Roll at the rate of 26 Dollars 66 Cents per annum, to comm
DATE: 1850-02-09 00:00:00

PAGE: 55
NAME PASSAGE: 10836. A. New York. Sarah Torry Widow of Amos Torry who was a pensioner under the Act of and who died on the 5 August 1831 of Tompkins in the State of N. York who was a Private in the Rev. C
DATE: 18

In [88]:
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None
):
    display(
        award_history.loc[
            award_history["file_id"].isin(first_files),
            [
                "file_id", "page_number", "name_passage",
                "certificate_date_draft"
            ]
        ]
    )

,file_id,page_number,name_passage,certificate_date_draft
0,111403885,2,"Reported 4683. New York Albany Abigail Torrey widow of John Torrey Con who was a Private in the Revolution Inscribed on the Roll at the rate of 27 Dollars 22 Cents per annum, to commence on the 4th day of",1844-01-25
1,111403885,32,"New York 7877 Abigail Torrey Abigail Torrey widow of John Torrey who was a pensioner under the Act of and who died on the 9 March, 1821- of Seneca (in the State of N. Yk) who was a private in the compan",1843-11-29
2,111403929,4,"3332 A New York. Sarah Torry widow of Amos Torry who served in the Revolutionary war, as a Private for 8 months Inscribed on the Roll at the rate of 26 Dollars 66 Cents per annum, to comm",1850-02-09
3,111403929,55,10836. A. New York. Sarah Torry Widow of Amos Torry who was a pensioner under the Act of and who died on the 5 August 1831 of Tompkins in the State of N. York who was a Private in the Rev. C,1830-02-09
4,111403929,56,"Republished 5335. 104. New York. Sarah Torry widow of Amos Torry Conn. who served in the Revolutionary war, as a Private for 15 months Inscribed on the Roll at the rate of 51 Dollars 21 Cents per annum,",1850-02-16
5,111403929,57,"8752 A. New York. Sarah Torry widow of Amos Torry who was a Private in the Revolution for 8 months Inscribed on the Roll at the rate of 26 Dollars 66 Cents per annum, to commence on the 4t",NaT
6,111403929,58,10836. A. New York. Sarah Torry Widow of Amos Torry who was a pensioner under the Act of and who died on the 5 th Aug t 1831 of Tipton Co in the State of N. York who was a Private in the Com,1850-02-16
7,111403929,59,"8752. A. New York Sarah Torry widow of Amos Torry who was a Private in the Revolution for 18 months Inscribed on the Roll at the rate of 51 Dollars 21 Cents per annum, to commence on the 4",1837-07-16
8,111404093,2,"New York 3738 Abigail Torrey widow of Jonathan Torrey who was a Private in the Massachusetts line for 12 months Inscribed on the Roll at the rate of Fifty — Dollars — Cents per annum, to",1843-12-12
9,111404093,28,"ril 1838, from 4 Sep 1839 to 4 th March 1840. Agt notified 11. Decr 1840 Albany 2629 Abigail Torrey widow of Jonathan Torrey who was a pensioner under the Act of March 1818 and who died on the 14. June 1837 of Madison Co. in the State of N. York who was a Pr",1839-05-03


In [90]:
person_pairs = {
    "111403885": ("Abigail Torrey", "John Torrey"),
    "111403929": ("Sarah Torry", "Amos Torry"),
    "111404093": ("Abigail Torrey", "Jonathan Torrey")
}

for file_id, (widow, veteran) in person_pairs.items():
    mask = award_history["file_id"].astype(str).eq(file_id)

    award_history.loc[mask, "widow_name_working"] = widow
    award_history.loc[mask, "veteran_name_working"] = veteran
    award_history.loc[mask, "person_id_draft"] = f"{file_id}_widow_01"
    award_history.loc[mask, "identity_status"] = (
        "Provisional link from OCR name passages"
    )

award_history.to_csv(
    "search_results/widow_award_history_draft.csv",
    index=False
)

In [91]:
widow_summary = (
    award_history
    .dropna(subset=["person_id_draft"])
    .groupby("person_id_draft")
    .agg(
        widow_name=("widow_name_working", "first"),
        veteran_name=("veteran_name_working", "first"),
        candidate_pages=("file_id", "size"),
        parsed_dates=("certificate_date_draft", "count"),
        earliest_certificate_found=("certificate_date_draft", "min")
    )
)

widow_summary["unresolved_dates"] = (
    widow_summary["candidate_pages"] - widow_summary["parsed_dates"]
)

widow_summary["first_award_status"] = "Not established"

widow_summary.to_csv(
    "search_results/widow_summary_draft.csv"
)

display(widow_summary)

,widow_name,veteran_name,candidate_pages,parsed_dates,earliest_certificate_found,unresolved_dates,first_award_status
person_id_draft,,,,,,,
111403885_widow_01,Abigail Torrey,John Torrey,2,2,1843-11-29,0,Not established
111403929_widow_01,Sarah Torry,Amos Torry,6,5,1830-02-09,1,Not established
111404093_widow_01,Abigail Torrey,Jonathan Torrey,2,2,1839-05-03,0,Not established


In [92]:
import re

def extract_name_evidence(text):
    text = " ".join(str(text).split())

    matches = list(re.finditer(
        r"\bwidow\s+of\b", text, flags=re.I
    ))

    if len(matches) != 1:
        return pd.Series({
            "widow_name_context": pd.NA,
            "veteran_name_candidate": pd.NA,
            "name_pair_status": "Missing or multiple widow-of passages"
        })

    match = matches[0]

    before = text[max(0, match.start() - 100):match.start()]
    after = text[match.end():]

    veteran = re.match(
        r"\s*(.{1,100}?)\s+who\b",
        after,
        flags=re.I
    )

    return pd.Series({
        "widow_name_context": before.strip(),
        "veteran_name_candidate": (
            veteran.group(1).strip(" ,.;")
            if veteran else pd.NA
        ),
        "name_pair_status": "Needs name-boundary review"
    })

name_evidence = (
    award_history["text_original"]
    .apply(extract_name_evidence)
)

for column in name_evidence.columns:
    award_history[column] = name_evidence[column]

display(award_history[
    ["widow_name_context", "veteran_name_candidate"]
].head(10))

award_history.to_csv(
    "search_results/widow_award_history_draft.csv",
    index=False
)

,widow_name_context,veteran_name_candidate
0,Reported 4683. New York Albany Abigail Torrey,John Torrey Con
1,New York 7877 Abigail Torrey Abigail Torrey,John Torrey
2,3332 A New York. Sarah Torry,Amos Torry
3,10836. A. New York. Sarah Torry,Amos Torry
4,Republished 5335. 104. New York. Sarah Torry,Amos Torry Conn
5,8752 A. New York. Sarah Torry,Amos Torry
6,10836. A. New York. Sarah Torry,Amos Torry
7,8752. A. New York Sarah Torry,Amos Torry
8,New York 3738 Abigail Torrey,Jonathan Torrey
9,"ril 1838, from 4 Sep 1839 to 4 th March 1840. ...",Jonathan Torrey


In [93]:
print("Total pages:", len(award_history))
print(
    "Pages with a veteran-name candidate:",
    award_history["veteran_name_candidate"].notna().sum()
)

name_check = award_history.loc[
    award_history["veteran_name_candidate"].notna(),
    ["widow_name_context", "veteran_name_candidate", "page_url"]
].sample(n=20, random_state=2026)

with pd.option_context("display.max_colwidth", None):
    display(name_check)

Total pages: 40083
Pages with a veteran-name candidate: 39010


,widow_name_context,veteran_name_candidate,page_url
5754,"Connecticut 6987 Reported Ruth Booth, who was",Isaac Jones,https://catalog.archives.gov/id/144080290?objectPage=19
5065,n Hyde Morristown St. Lawrence Co N.Y. Repeated July 3/49 215086 2767. New York Albany Rebecca Hyde,Ephraim Hyde,https://catalog.archives.gov/id/144036669?objectPage=4
35641,Surrendered & cancelled & new Certificate issued at P70 - (Pw. Da t notified) 970 Name Ann M. Gould,Daniel,https://catalog.archives.gov/id/54700227?objectPage=63
3589,Bounty Land 26875-100 1853 986 Rhode Island Marey Wilkinson,R.D. William Wilkinson,https://catalog.archives.gov/id/111713363?objectPage=3
7350,4639 North Carolina Mrs. McCulloch (,Robert McCulloch),https://catalog.archives.gov/id/144177825?objectPage=2
9041,4125 Massachusetts Reported Hannah Sturtevant,Isaac Sturtevant,https://catalog.archives.gov/id/144282363?objectPage=25
5257,Albany 5603 Elizabeth Jacobs,Cornelius Jacobs N.Y,https://catalog.archives.gov/id/144046989?objectPage=15
3755,"2 Cts. 1 of Res? 1 One acts 238 44 Rep d Act Richmond 7028. Virginia. Wilmouth Williams,",James M. Williams,https://catalog.archives.gov/id/111722709?objectPage=32
20950,Albany 829 Agatha Amelin,Baptist Amelin N.Y,https://catalog.archives.gov/id/53814724?objectPage=13
38219,8801 N. Tennessee Margaret Nendricks,Albert Nendricks,https://catalog.archives.gov/id/54851532?objectPage=22


In [94]:
def veteran_name_check(value):
    if pd.isna(value):
        return "Missing candidate"

    words = re.findall(r"[A-Za-z]+", value)

    if len(words) < 2:
        return "Single-word candidate — review"

    return "Multiple-word candidate — unverified"

award_history["veteran_name_check"] = (
    award_history["veteran_name_candidate"]
    .apply(veteran_name_check)
)

print(
    award_history["veteran_name_check"]
    .value_counts()
    .to_string()
)

award_history.to_csv(
    "search_results/widow_award_history_draft.csv",
    index=False
)

veteran_name_check
Multiple-word candidate — unverified    35569
Single-word candidate — review           3441
Missing candidate                        1073


In [95]:
for index in name_check.head(5).index:
    page = award_history.loc[index]
    text = page["text_original"]

    match = re.search(r"\bwidow\s+of\b", text, flags=re.I)

    print("\nSOURCE:", page["page_url"])
    if match:
        print(repr(text[
            max(0, match.start() - 180):
            match.end() + 100
        ]))


SOURCE: https://catalog.archives.gov/id/144080290?objectPage=19
'Connecticut 6987\nReported\nRuth Booth, who was widow of Isaac Jones\nwho was a pensioner under the Act of and who died on the 6 th August 1793 of Fairfield '

SOURCE: https://catalog.archives.gov/id/144036669?objectPage=4
'certif Duplicate, issued Aug 19 1849 and sent to Freeman Hyde Morristown St. Lawrence Co N.Y.\nRepeated July 3/49\n215086\n2767. New York Albany Rebecca Hyde widow of Ephraim Hyde who served in the Revolutionary war, as a private\nInscribed on the Roll at the rate of'

SOURCE: https://catalog.archives.gov/id/54700227?objectPage=63
'1853 Sept. 13 - Certificate of 20. July Surrendered & cancelled & new Certificate issued at P70 - (Pw. Da t notified)\n970 Name Ann M. Gould widow of Daniel who served in the Revolutionary war, as a Private\nInscribed on the Roll at the rate of 20 70'

SOURCE: https://catalog.archives.gov/id/111713363?objectPage=3
'Bounty Land 26875-100 1853\n986 Rhode Island\nMarey Wilkins

In [96]:
case_timeline = (
    file_dates
    .groupby("earliest_date_period_draft")
    .agg(
        cases=("candidate_pages", "size"),
        cases_with_unresolved_dates=(
            "unresolved_dates",
            lambda values: values.gt(0).sum()
        )
    )
    .sort_index()
)

display(case_timeline)

case_timeline.to_csv(
    "search_results/case_timeline_draft.csv"
)

,cases,cases_with_unresolved_dates
earliest_date_period_draft,,
1836-07-04 to 1838-07-06,1822,230
1838-07-07 to 1843-03-02,4165,679
1843-03-03 to 1844-06-16,3114,1949
1844-06-17 to 1848-02-01,2374,732
1848-02-02 to 1848-07-28,225,79
1848-07-29 to 1853-02-02,1702,561
1853-02-03 onward,2992,522
Before 1836-07-04 — review separately,861,227
Date unresolved,2936,2936
